# Space2Soil: Phase 4 Multi-Criteria Recommendation Engine
### Algorithmic Evaluation of Climate-Resilient Crop Rotation Strategies
**Project:** Space2Soil (NASA Space Apps Challenge: "Field Shift: Adapting Farms with NASA Data")  
**Location:** Ejura, Ashanti Region, Ghana (`7.3856° N, -1.3562° W`)

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

rec_dir = os.path.abspath(os.path.join('..', 'data', 'processed', 'recommendations'))
df_scenarios = pd.read_csv(os.path.join(rec_dir, 'rotation_scenarios_comparison.csv'))
print('Loaded scenarios comparison table:')
df_scenarios[['rank', 'scenario_name', 'composite_score', 'net_profit_ghs', 'net_soil_nitrogen_balance_kg_ha', 'return_on_investment_roi']]

## 1. Multi-Criteria Radar Comparison (The 4 Evaluation Dimensions)

In [ ]:
categories = ['Climate Resilience', 'Soil Health', 'Economic Profit', 'Nutrition Diversity']
N = len(categories)

# Radar angles
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

plt.xticks(angles[:-1], categories, fontweight='bold', size=12)
ax.set_rlabel_position(0)
plt.yticks([25, 50, 75, 100], ['25', '50', '75', '100'], color='grey', size=10)
plt.ylim(0, 105)

colors = {'Baseline': '#e41a1c', 'Scenario_1': '#2ca02c', 'Scenario_2': '#377eb8'}
labels = {
    'Baseline': 'Baseline: Monoculture Maize (Score: 51.2)',
    'Scenario_1': 'Scenario 1: Regenerative Resilience (Score: 97.1)',
    'Scenario_2': 'Scenario 2: Commercial Diversification (Score: 89.1)'
}

for idx, row in df_scenarios.iterrows():
    s_key = row['scenario_key']
    values = [
        row['climate_resilience_subscore'],
        row['soil_health_subscore'],
        row['economic_profit_subscore'],
        row['nutritional_diversity_subscore']
    ]
    values += values[:1]
    
    ax.plot(angles, values, linewidth=2.5, linestyle='solid', label=labels[s_key], color=colors[s_key])
    ax.fill(angles, values, color=colors[s_key], alpha=0.15)

plt.title('Multi-Criteria Performance Across 4 Core Evaluation Dimensions', size=14, fontweight='bold', y=1.08)
plt.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))
plt.show()

## 2. Economic Balance: Revenue vs Costs vs Net Profit (GHS / Hectare)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

scenarios = df_scenarios['scenario_name'].str.split('(').str[0]
x = np.arange(len(scenarios))
width = 0.25

rects1 = ax.bar(x - width, df_scenarios['gross_revenue_ghs'], width, label='Gross Revenue (GHS/ha)', color='#4daf4a')
rects2 = ax.bar(x, df_scenarios['total_production_cost_ghs'], width, label='Production Cost (GHS/ha)', color='#e41a1c')
rects3 = ax.bar(x + width, df_scenarios['net_profit_ghs'], width, label='Net Farm Profit (GHS/ha)', color='#377eb8')

ax.set_ylabel('Ghanaian Cedis (GHS / ha)', fontweight='bold')
ax.set_title('Agro-Economic Comparison: Revenue, Inputs and Net Smallholder Profit', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontweight='bold')
ax.legend()

# Add value labels on profit bars
for rect in rects3:
    h = rect.get_height()
    ax.annotate(f'GHS {h:,.0f}',
                xy=(rect.get_x() + rect.get_width() / 2, h),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Soil Regeneration: Net Nitrogen Balance & Ground Cover Duration

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Nitrogen balance
n_colors = ['#2ca02c' if n >= 0 else '#d95f02' for n in df_scenarios['net_soil_nitrogen_balance_kg_ha']]
ax1.bar(scenarios, df_scenarios['net_soil_nitrogen_balance_kg_ha'], color=n_colors, width=0.45, edgecolor='black')
ax1.axhline(0, color='black', linewidth=1)
ax1.set_ylabel('Net Soil Nitrogen (kg N / ha / year)', fontweight='bold')
ax1.set_title('Soil Biological Nitrogen Balance', fontweight='bold')

# Ground cover
ax2.bar(scenarios, df_scenarios['soil_cover_duration_months'], color='#1f77b4', width=0.45, edgecolor='black')
ax2.set_ylabel('Living Ground Cover (Months / Year)', fontweight='bold')
ax2.set_title('Soil Cover Duration (Erosion Protection Against MFI > 140)', fontweight='bold')
ax2.set_ylim(0, 12)

plt.tight_layout()
plt.show()

## 4. Farmer Action Advisory Cards (Generated via Explainable AI)

In [ ]:
cards_path = os.path.join(rec_dir, 'farmer_recommendation_cards.json')
with open(cards_path, 'r', encoding='utf-8') as f:
    cards = json.load(f)

for card in cards:
    print('='*75)
    print(f"{card['title'].upper()} -- RANK #{card['composite_rank']} (Score: {card['composite_score']})")
    print(f"VERDICT: {card['verdict']}")
    print('-'*75)
    if 'why_it_wins' in card:
        print('KEY BENEFITS:')
        for b in card['why_it_wins']:
            print(f'  * {b}')
    if 'nasa_decision_triggers' in card:
        print('\nNASA DATA TRIGGERS:')
        for k, v in card['nasa_decision_triggers'].items():
            print(f'  [{k.upper()}]: {v}')
    print(f"FINANCIAL SUMMARY: Net Profit = GHS {card['financial_return']['net_profit_ghs']:,} (ROI: {card['financial_return']['roi']}x)\n")